# 08 – Event-to-Lake Audit (HMAGLOFDB)

In notebook 07, I matched historical GLOF events to lakes by distance only. A close match is not proof that it is the same lake: some events came from temporary (ephemeral) ice-dammed lakes that no longer exist, or from lakes that were never mapped.

In this notebook I **grade every event–lake match** before using any of them as labels. Lakes without an event are marked **unknown**, not safe.

### 1. Load data

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import BallTree

PROJECT_ROOT = Path("..")
hma = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "HMAGLOFDB.csv", encoding="latin1", sep=None, engine="python")
lakes = pd.read_csv(PROJECT_ROOT / "data" / "final" / "glofguard_training_data.csv")

pk = hma[hma["Country"].astype(str).str.strip().str.lower() == "pakistan"].copy()
pk = pk.dropna(subset=["Lat_lake", "Lon_lake"])
print("Pakistan events with coordinates:", len(pk))

Pakistan events with coordinates: 151


### 2. Find the nearest and second-nearest lake for each event

If two lakes are almost equally close, the match is ambiguous.

In [8]:
tree = BallTree(np.radians(lakes[["latitude", "longitude"]].values), metric="haversine")
dist, idx = tree.query(np.radians(pk[["Lat_lake", "Lon_lake"]].values), k=2)
dist_km = dist * 6371

pk["matched_sample_id"] = lakes.iloc[idx[:, 0]]["sample_id"].values
pk["match_km"] = dist_km[:, 0].round(3)
pk["second_km"] = dist_km[:, 1].round(3)
pk["lake_id_status"] = np.where(pk["GL_ID"].astype(str).str.startswith("GL"), "mapped",
                       np.where(pk["GL_ID"].astype(str).str.strip().str.lower() == "ephemeral", "ephemeral", "not mapped"))
pk["lake_id_status"].value_counts()

lake_id_status
ephemeral     107
mapped         23
not mapped     21
Name: count, dtype: int64

### 3. Grade each match

- **strong**: the event lake has a real lake ID (mapped), is within 1 km, and no other lake is almost as close
- **possible**: mapped lake within 3 km, or a close but ambiguous match
- **weak**: ephemeral or unmapped lake (it may not exist anymore), or more than 3 km away. Not used as a label.

In [9]:
def grade(r):
    if r["lake_id_status"] != "mapped" or r["match_km"] > 3:
        return "weak"
    clear = r["second_km"] >= 2 * r["match_km"]
    if r["match_km"] <= 1 and clear:
        return "strong"
    return "possible"

pk["match_grade"] = pk.apply(grade, axis=1)
print(pk["match_grade"].value_counts())
pk.loc[pk["match_grade"] != "weak",
       ["Year_exact", "Lake_name", "GL_ID", "Lake_type", "matched_sample_id", "match_km", "second_km", "match_grade"]
      ].sort_values("match_km")

match_grade
weak        141
possible      6
strong        4
Name: count, dtype: int64


,Year_exact,Lake_name,GL_ID,Lake_type,matched_sample_id,match_km,second_km,match_grade
732,2024.0,Unknown,GL072176E35781N,Moraine dammed,812,0.030,0.459,strong
376,1999.0,Khalti,GL073359E36189N,Moraine dammed,1755,0.046,0.909,strong
350,1993.0,Unnamed,GL073193E36038N,Moraine dammed,1163,0.061,0.509,strong
2,1661.0,Unnamed,GL075907E35687N,Unknown,4356,0.201,0.286,possible
39,1895.0,Unnamed,GL075907E35687N,Unknown,4356,0.201,0.286,possible
31,1890.0,Unnamed,GL075907E35687N,Unknown,4356,0.201,0.286,possible
280,1980.0,Unnamed,GL073423E36117N,Moraine dammed,1509,0.386,0.921,strong
359,1994.0,Sosot,GL073196E36026N,Moraine dammed,1822,0.860,1.194,possible
409,2005.0,Sosot,GL073196E36026N,Moraine dammed,1822,0.860,1.194,possible
453,2012.0,Sosot,GL073196E36026N,Moraine dammed,1822,0.860,1.194,possible


### 4. One label status per lake

- **confirmed_event**: at least one strong match
- **possible_event**: only possible matches
- **unknown**: every other lake (no recorded event does **not** mean safe)

In [10]:
best = (pk[pk["match_grade"] != "weak"]
        .assign(rank=lambda d: d["match_grade"].map({"strong": 0, "possible": 1}))
        .sort_values("rank")
        .groupby("matched_sample_id")
        .agg(event_count=("match_grade", "size"), best_grade=("match_grade", "first")))

lakes["event_status"] = "unknown"
lakes.loc[lakes["sample_id"].isin(best.index[best["best_grade"] == "strong"]), "event_status"] = "confirmed_event"
lakes.loc[lakes["sample_id"].isin(best.index[best["best_grade"] == "possible"]), "event_status"] = "possible_event"

print(lakes["event_status"].value_counts())
print()
print("Were these lakes high-risk (label 1) in my training data?")
print(pd.crosstab(lakes["event_status"], lakes["label"]))

event_status
unknown            8800
confirmed_event       4
possible_event        2
Name: count, dtype: int64

Were these lakes high-risk (label 1) in my training data?
label               0    1
event_status              
confirmed_event     0    4
possible_event      0    2
unknown          8455  345


### 5. Decision: train on real labels, or use them for testing?

In [11]:
n_confirmed = (lakes["event_status"] == "confirmed_event").sum()
n_possible = (lakes["event_status"] == "possible_event").sum()
print("Confirmed event lakes:", n_confirmed)
print("Possible event lakes:", n_possible)

MIN_POSITIVES = 30  # rough rule of thumb, not a fixed standard
if n_confirmed >= MIN_POSITIVES:
    print("Enough confirmed lakes to try training a model on real event labels.")
else:
    print("Too few confirmed lakes to train a reliable model.")
    print("Keep the current model (proxy labels) and use confirmed lakes as a clean test set.")

Confirmed event lakes: 4
Possible event lakes: 2
Too few confirmed lakes to train a reliable model.
Keep the current model (proxy labels) and use confirmed lakes as a clean test set.


### 6. Save the audited labels

This is the only file this notebook saves. It keeps every event with its grade, so the matching can be checked by anyone.

In [12]:
out_dir = PROJECT_ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
pk[["GF_ID", "Year_exact", "Lake_name", "Glacier_name", "GL_ID", "Lake_type", "Lat_lake", "Lon_lake",
    "matched_sample_id", "match_km", "second_km", "lake_id_status", "match_grade"]
  ].to_csv(out_dir / "event_lake_audit.csv", index=False)
print("Saved: data/processed/event_lake_audit.csv")

Saved: data/processed/event_lake_audit.csv


### 7. Clean test on confirmed lakes

In [13]:
import joblib
from sklearn.base import clone

saved = joblib.load(PROJECT_ROOT / "artifacts" / "final_model.joblib")
final_model, feature_cols = saved["pipeline"], saved["features"]

# Same 0.25 degree grid cells as notebooks 06 and 07
lakes["geo_group"] = ((lakes["latitude"] // 0.25).astype(int).astype(str) + "_" +
                      (lakes["longitude"] // 0.25).astype(int).astype(str))

test_ids = lakes.loc[lakes["event_status"] != "unknown", "sample_id"]
held = lakes.loc[lakes["sample_id"].isin(test_ids), "geo_group"].unique()

# Retrain WITHOUT the grid cells of these lakes, so they are unseen
train = lakes[~lakes["geo_group"].isin(held)]
model = clone(final_model).fit(train[feature_cols], train["label"])

lakes["score"] = model.predict_proba(lakes[feature_cols])[:, 1]
lakes["percentile_%"] = (lakes["score"].rank(pct=True) * 100).round(1)

result = (lakes[lakes["event_status"] != "unknown"]
          [["sample_id", "event_status", "score", "percentile_%"]]
          .sort_values("percentile_%", ascending=False))
result["in_top_20%"] = result["percentile_%"] >= 80
print("Held-out grid cells:", len(held), "| training lakes:", len(train))
result

Held-out grid cells: 4 | training lakes: 8550


,sample_id,event_status,score,percentile_%,in_top_20%
4355,4356,possible_event,0.843974,94.8,True
811,812,confirmed_event,0.298491,67.1,False
1754,1755,confirmed_event,0.293431,66.6,False
1508,1509,confirmed_event,0.239777,58.7,False
1162,1163,confirmed_event,0.173589,52.6,False
1821,1822,possible_event,0.160328,50.1,False


### Conclusion

I audited all 151 GLOF events recorded for Pakistan in HMAGLOFDB before using any of them as labels. Most events could not be linked to a lake in my dataset: 107 came from ephemeral (temporary) lakes that no longer exist, and 21 came from lakes that were never mapped. After grading every match, only 4 lakes had a strong match (812, 1755, 1163, 1509) and 2 had a possible match (4356, 1822). All other lakes are marked unknown, not safe.

All 6 of these lakes were already labelled high-risk in my training data, so my proxy labels agree with the real events. However, 6 lakes are far too few to train a reliable model, so I kept my current model and used these lakes as a clean test set.

For a fair test, I retrained my model without the grid cells containing these lakes and scored them as unseen data:

| Lake | Status | Percentile |
|---|---|---|
| 4356 | possible | 94.8% |
| 812 | confirmed | 67.1% |
| 1755 | confirmed | 66.6% |
| 1509 | confirmed | 58.7% |
| 1163 | confirmed | 52.6% |
| 1822 | possible | 50.1% |

All 6 lakes ranked above the middle (50th percentile), which would be unlikely by chance (about 1 in 64). This suggests my model captures real risk patterns. However, only one lake reached the top 20%, so the model ranks real GLOF lakes above average but does not strongly single them out. This also shows that part of the stronger result in notebook 07 came from weaker matches.

Because this test has only 6 lakes, it supports my model but does not prove it. I therefore present GLOFGuard as a regional GLOF risk watch-list, not a tool that predicts exactly which lake will burst.